In [4]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, in_channels):
        super(Encoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.mid_level = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 192, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x1 = self.encoder(x)
        x2 = self.mid_level(x1)
        return x2

In [41]:
import torch
import torch.nn as nn

from groupy.gconv.pytorch_gconv.splitgconv2d import P4ConvZ2, P4ConvP4
from groupy.gconv.pytorch_gconv import P4MConvZ2, P4MConvP4M
from groupy.gconv.pytorch_gconv.pooling import plane_group_spatial_max_pooling

class GEncoder(nn.Module):
    def __init__(self, in_channels):
        super(GEncoder, self).__init__()

        self.conv1 = P4MConvZ2(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = P4MConvP4M(32, 32, kernel_size=3, stride=1, padding=1)

        self.conv3 = P4MConvP4M(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv4 = P4MConvP4M(64, 128, kernel_size=3, stride=1, padding=1)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = plane_group_spatial_max_pooling(x, 2, 2)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        xs = x.size()
        x = x.view(xs[0], xs[1] * xs[2], xs[3], xs[4])
        return x

In [42]:
from torchsummary import summary
enc = GEncoder(1).cuda()
summary(enc, (1, 128, 128))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
         P4MConvZ2-1      [-1, 32, 8, 128, 128]             320
        P4MConvP4M-2      [-1, 32, 8, 128, 128]          73,760
        P4MConvP4M-3        [-1, 64, 8, 64, 64]         147,520
        P4MConvP4M-4       [-1, 128, 8, 64, 64]         589,952
Total params: 811,552
Trainable params: 811,552
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.06
Forward/backward pass size (MB): 112.00
Params size (MB): 3.10
Estimated Total Size (MB): 115.16
----------------------------------------------------------------


In [43]:
class Decoder(nn.Module):
    def __init__(self, out_channels):
        super(Decoder, self).__init__()
        
        self.decoder = nn.Sequential(
            nn.Conv2d(128*8, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(192, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, out_channels, kernel_size=2, stride=2, output_padding=0),
        )
        
    def forward(self, x):
        return self.decoder(x)

In [37]:
# class GDecoder(nn.Module):
#     def __init__(self, out_channels):
#         super(GDecoder, self).__init__()
        
#         self.conv5 = P4MConvP4M(128, 64, kernel_size=3, padding=1)
#         self.conv6 = P4MConvP4M(64, 32, kernel_size=3, padding=1)

#         self.convt = nn.ConvTranspose2d(32*8, out_channels, kernel_size=2, stride=2, output_padding=0)
        
#     def forward(self, x):
#         x = F.relu(self.conv5(x))
#         x = F.relu(self.conv6(x))
#         xs = x.size()
#         x = x.view(xs[0], xs[1] * xs[2], xs[3], xs[4])
#         x = self.convt(x)

#         return x

In [ ]:
from torchsummary import summary
dec = Decoder(1).cuda()
# summary(dec, (1, 128, 128))

In [45]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import imageio
import skimage

class ERSSLDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)

    def __len__(self):
        return sum(len(os.listdir(os.path.join(self.root_dir, cls, 'images'))) for cls in self.classes)

    def __getitem__(self, idx):
        current_class_idx = 0
        while idx >= len(os.listdir(os.path.join(self.root_dir, self.classes[current_class_idx], 'images'))):
            idx -= len(os.listdir(os.path.join(self.root_dir, self.classes[current_class_idx], 'images')))
            current_class_idx += 1

        current_class = self.classes[current_class_idx]
        img_folder = os.path.join(self.root_dir, current_class, 'images')
#         mask_folder = os.path.join(self.root_dir, current_class, 'updated_masks')

        img_name = os.listdir(img_folder)[idx]
        img_path = os.path.join(img_folder, img_name)
#         mask_name = os.path.splitext(img_name)[0] + '_mask.png'  # Assuming mask files have the same name as images with '_mask' appended
#         mask_path = os.path.join(mask_folder, mask_name)

#         image = Image.open(img_path)#.convert("RGB")
        image = imageio.imread(img_path)
        aug_image = skimage.util.random_noise(image)
#         image = np.stack((image, noisy_image),axis=2)
        image = image.astype('float32')
        aug_image = image.astype('float32')

        if self.transform:
            image = self.transform(image)
#             mask = self.transform(mask)
            aug_image = self.transform(aug_image)


        return image, aug_image

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import torch.nn.functional as F


import numpy as np

np.random.seed(34)
torch.manual_seed(34)

enc1 = GEncoder(1)
enc2 = GEncoder(1)
# in_channels = 1
# out_channels = 1  # Assuming binary segmentation
# model = NNet(in_channels, out_channels)

# Define your loss function and optimizer
# criterion = nn.BCEWithLogitsLoss()
# tv = TVLoss()
# criterion = nn.CosineEmbeddingLoss()

def loss_func(feat1, feat2):
    # minimize average magnitude of cosine similarity
    return F.cosine_similarity(feat1, feat2).mean()


optimizer1 = optim.AdamW(enc1.parameters(), lr=7e-4, weight_decay=1e-4)
optimizer2 = optim.AdamW(enc2.parameters(), lr=7e-4, weight_decay=1e-4)

transform = transforms.Compose([
    transforms.ToTensor()
])


# Define your dataset

root_dir = '/localhome/asa420/MIAL/data/confocal-data/vess_enh_unet/'

dataset = ERSSLDataset(root_dir, transform=transform)


# Assuming an 80-20 train-test split
train_size = int(0.85 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


batch_size = 16
tar_tensor = torch.ones(batch_size)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Training loop
num_epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

enc1.to(device)
enc2.to(device)

for epoch in range(num_epochs):
    enc1.train()
    enc2.train()
    for inputs, aug_input in train_loader:
        inputs, aug_input = inputs.to(device), aug_input.to(device)

        # Forward pass
        outputs1 = enc1(inputs)
        outputs2 = enc2(aug_input)

#         loss = criterion(outputs1, outputs2, target=tar_tensor)# + tv(masks)
        loss = loss_func(outputs1, outputs2)

        # Backward pass and optimization
        optimizer1.zero_grad()
        optimizer2.zero_grad()
        
        loss.backward()
        
        optimizer1.step()
        optimizer2.step()

    # Print the training loss for each epoch
    print(f"Epoch [{epoch + 1}/{num_epochs}], Training Loss: {loss.item()}")

# Testing loop
enc1.eval()
enc2.eval()

test_loss = 0.0
with torch.no_grad():
    for inputs, aug_input in test_loader:
        inputs, aug_input = inputs.to(device), aug_input.to(device)

        # Forward pass
        outputs1 = enc1(inputs)
        outputs2 = enc2(aug_input)

#         loss = criterion(outputs1, outputs2, target=tar_tensor)
        loss = loss_func(outputs1, outputs2)
        test_loss += loss.item()

# Calculate and print the average test loss
average_test_loss = test_loss / len(test_loader)
print(f"Average Test Loss: {average_test_loss}")

# Save the trained model
# torch.save(model.state_dict(), 'NNet_groupy_p4m_v2_STED_noise.pth')
torch.save(enc1.state_dict(), 'Genc1.pth')
torch.save(enc2.state_dict(), 'Genc2.pth')

Epoch [1/20], Training Loss: 1.5854812573934396e-08
Epoch [2/20], Training Loss: 0.0
Epoch [3/20], Training Loss: 0.0
Epoch [4/20], Training Loss: 0.0
Epoch [5/20], Training Loss: 0.0
Epoch [6/20], Training Loss: 0.0
Epoch [7/20], Training Loss: 0.0


KeyboardInterrupt: 

In [48]:
dec = Decoder(1).cuda()

In [49]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import imageio
import skimage

class ERDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)

    def __len__(self):
        return sum(len(os.listdir(os.path.join(self.root_dir, cls, 'images'))) for cls in self.classes)

    def __getitem__(self, idx):
        current_class_idx = 0
        while idx >= len(os.listdir(os.path.join(self.root_dir, self.classes[current_class_idx], 'images'))):
            idx -= len(os.listdir(os.path.join(self.root_dir, self.classes[current_class_idx], 'images')))
            current_class_idx += 1

        current_class = self.classes[current_class_idx]
        img_folder = os.path.join(self.root_dir, current_class, 'images')
        mask_folder = os.path.join(self.root_dir, current_class, 'updated_masks')

        img_name = os.listdir(img_folder)[idx]
        img_path = os.path.join(img_folder, img_name)
        mask_name = os.path.splitext(img_name)[0] + '_mask.png'  # Assuming mask files have the same name as images with '_mask' appended
        mask_path = os.path.join(mask_folder, mask_name)

#         image = Image.open(img_path)#.convert("RGB")
        mask = Image.open(mask_path)#.convert("L")  # Convert to grayscale mask
#         image = imageio.imread(img_path)
#         mask = imageio.imread(mask_path)
#         print(image.shape)
#         noisy_image = skimage.util.random_noise(image)
#         image = np.concatenate((image, noisy_image))
#         image = np.stack((image, noisy_image),axis=2)
#         print(image.shape)
#         image = Image.fromarray(image)
#         image = image.astype('float32')
        image = Image.open(img_path)
        
#         mask[mask==0] = 100
#         mask[mask==255] = 0
#         mask[mask==100] = 255

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)


        return image, mask

In [50]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import torch.nn.functional as F


import numpy as np

np.random.seed(34)
torch.manual_seed(34)

# enc1 = Encoder(1)
# enc2 = Encoder(1)
# in_channels = 1
# out_channels = 1  # Assuming binary segmentation
# model = NNet(in_channels, out_channels)

# Define your loss function and optimizer
bceloss = nn.BCEWithLogitsLoss()
# tv = TVLoss()
optimizer = optim.AdamW(dec.parameters(), lr=7e-4, weight_decay=1e-4)


def loss_func(feat1, feat2):
    # minimize average magnitude of cosine similarity
    return F.cosine_similarity(feat1, feat2).mean()


transform = transforms.Compose([
    transforms.ToTensor()
])


# Define your dataset

root_dir = '/localhome/asa420/MIAL/data/sted-data/vess_enh_unet/'

dataset = ERDataset(root_dir, transform=transform)


# Assuming an 80-20 train-test split
train_size = int(0.85 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


batch_size = 16
tar_tensor = torch.ones(batch_size)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Training loop
num_epochs = 200
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dec.to(device)

for epoch in range(num_epochs):
    dec.train()
    
    for inputs, masks in train_loader:
        inputs, masks = inputs.to(device), masks.to(device)

        # Forward pass
        embeddings = enc1(inputs)
        outputs = dec(embeddings)

        loss = bceloss(outputs, masks)

        # Backward pass and optimization
        optimizer.zero_grad()
        
        loss.backward()
        
        optimizer.step()

    # Print the training loss for each epoch
    print(f"Epoch [{epoch + 1}/{num_epochs}], Training Loss: {loss.item()}")

# Testing loop
dec.eval()

test_loss = 0.0
with torch.no_grad():
    for inputs, masks in train_loader:
        inputs, masks = inputs.to(device), masks.to(device)

        # Forward pass
        embeddings = enc1(inputs)
        outputs = dec(embeddings)

        loss = bceloss(outputs, masks)

        test_loss += loss.item()

# Calculate and print the average test loss
average_test_loss = test_loss / len(test_loader)
print(f"Average Test Loss: {average_test_loss}")

# Save the trained model
# torch.save(model.state_dict(), 'NNet_groupy_p4m_v2_STED_noise.pth')
# torch.save(enc1.state_dict(), 'enc1.pth')
# torch.save(enc2.state_dict(), 'enc2.pth')


Epoch [1/200], Training Loss: 5.597012519836426
Epoch [2/200], Training Loss: 3.180565357208252
Epoch [3/200], Training Loss: 2.084399700164795
Epoch [4/200], Training Loss: 0.9328719973564148
Epoch [5/200], Training Loss: 1.6661779880523682
Epoch [6/200], Training Loss: 0.6197354197502136
Epoch [7/200], Training Loss: 0.5198298692703247
Epoch [8/200], Training Loss: 0.5058680772781372
Epoch [9/200], Training Loss: 0.4778600335121155
Epoch [10/200], Training Loss: 0.47789251804351807
Epoch [11/200], Training Loss: 0.4741872549057007
Epoch [12/200], Training Loss: 0.4597621262073517
Epoch [13/200], Training Loss: 0.439083069562912
Epoch [14/200], Training Loss: 0.44998911023139954
Epoch [15/200], Training Loss: 0.45242202281951904
Epoch [16/200], Training Loss: 0.4152999520301819
Epoch [17/200], Training Loss: 0.44707292318344116
Epoch [18/200], Training Loss: 0.4289276599884033
Epoch [19/200], Training Loss: 0.44961321353912354
Epoch [20/200], Training Loss: 0.44135600328445435
Epoch [

KeyboardInterrupt: 

In [23]:
for inputs, masks in train_loader:
        inputs, masks = inputs.to(device), masks.to(device)

        # Forward pass
        embeddings = enc1(inputs)
        print(embeddings.shape)

torch.Size([16, 192, 64, 64])
torch.Size([13, 192, 64, 64])
